# Export an `.h5ad` subset of a selection and re-analyse it

<a href="https://colab.research.google.com/github/theislab/cellucid-python/blob/main/docs/user_guide/python_package/f_notebooks_tutorials/46_export_an_h5ad_subset_and_reanalyse.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Select a population in the viewer and take just those cells into a
fresh analysis: from a live viewer with `viewer.subset(adata, ...)`,
or with no notebook at all, by exporting the highlighted cells from
the web app as an `.h5ad` file that anndata and Scanpy read.

**Time:** 15 minutes. **You need:** `pip install cellucid scanpy
igraph leidenalg`. The same tutorial in R:
[Subset and re-analyse in R](../../r_package/f_notebooks_tutorials/12_subset_and_reanalyse.ipynb).

In [ ]:
import importlib.util

if importlib.util.find_spec("cellucid") is None:
    %pip install --quiet cellucid
if importlib.util.find_spec("scanpy") is None or importlib.util.find_spec("leidenalg") is None:
    %pip install --quiet scanpy igraph leidenalg

## The data

The 2,531-cell mouse pancreas (endocrinogenesis, day 15.5) AnnData checked
into the cellucid-python repository: UMAP in `obsm['X_umap_2d']`, cell types
in `obs['clusters']`, 27,998 genes. In a clone of the repository the cell
below finds it; anywhere else (Colab) it downloads it once. Use your own
`adata` in its place.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import anndata as ad

import cellucid

NAME = "endocrinogenesis_day15.5_raw.h5ad"
URL = (
    "https://raw.githubusercontent.com/theislab/cellucid-python/main/"
    "docs/user_guide/python_package/f_notebooks_tutorials/datasets/" + NAME
)


def pancreas_h5ad() -> Path:
    """The Pancreas file of a cellucid-python clone, else a copy downloaded once (35 MB)."""
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for candidate in (base / "datasets" / NAME, base / "docs/user_guide/python_package/f_notebooks_tutorials/datasets" / NAME):
            if candidate.is_file():
                return candidate
    if not (here / NAME).is_file():
        urlretrieve(URL, here / NAME)
    return here / NAME


DATA_PATH = pancreas_h5ad()
adata = ad.read_h5ad(DATA_PATH)
adata

## From a live viewer

Show the data, colour by `clusters`, lasso the `Fev+` and `Beta`
cells (the path into Beta cells) in **Highlighting** and click
**Confirm**, then run the next cell.

In [ ]:
DATASET_ID = "pancreas-e15-5"

viewer = cellucid.show_anndata(DATA_PATH, dataset_name="Pancreas E15.5", dataset_id=DATASET_ID)

In [ ]:
# which=None: the cells of the last selection, as a new AnnData.
branch = viewer.subset(adata) if viewer.selection is not None else None
branch

## Without a notebook: the web app's `.h5ad`

In the web app (www.cellucid.com, **Open… › File…**, the same file),
highlight the cells and click **.h5ad** beside **Clear**. The file
holds the highlighted cells of the active highlight page:

- their obs names and every obs field the viewer shows (your own
  label fields included), and each highlight group as a `bool` column;
- the genes, and the expression rows of the matrix the data mapping
  reads (`X`, a layer or `raw`), as a CSR `float32` matrix;
- the embeddings the viewer shows, in `obsm`.

Point `SUBSET` at it (the browser saved it to your Downloads folder;
in Colab, upload it).

In [ ]:
SUBSET = Path("Page_1_cells.h5ad")

if SUBSET.is_file():
    branch = ad.read_h5ad(SUBSET)
branch

Its obs names are the original dataset's, so it lines up with `adata`
(for example to bring back counts from a layer the export did not
include):

In [ ]:
assert branch.obs_names.isin(adata.obs_names).all()
counts = adata[branch.obs_names].copy()  # the full original object, these cells only
counts

## Re-analyse

Recompute the neighbourhood graph, the embedding and the clusters on
the subset alone; populations that were one cluster in the whole
dataset often split.

In [ ]:
import scanpy as sc

sub = counts.copy()
sc.pp.normalize_total(sub)
sc.pp.log1p(sub)
sc.pp.highly_variable_genes(sub, n_top_genes=1500)
sc.pp.pca(sub, n_comps=20)
sc.pp.neighbors(sub)
sc.tl.umap(sub)
sc.tl.leiden(sub, resolution=0.8, flavor="igraph", n_iterations=2, directed=False, key_added="sub_leiden")
sc.tl.rank_genes_groups(sub, "sub_leiden", method="wilcoxon")
sc.get.rank_genes_groups_df(sub, group=None).groupby("group", observed=True).head(3)

## Look at the subset

The re-analysed subset is an AnnData like any other, so the same one
line shows it, on its new UMAP:

In [ ]:
# The subset carried the whole dataset's UMAP as X_umap_2d, which the viewer
# would show first: replace it with the subset's own.
sub.obsm["X_umap_2d"] = sub.obsm["X_umap"]
sub_viewer = cellucid.show_anndata(sub, dataset_name="Pancreas E15.5: subset", dataset_id="pancreas-e15-5-subset")

In [ ]:
sub_viewer.stop()
viewer.stop()

## Next

- [Subsampled views](47_subsampled_views.ipynb): pull from a viewer
  that opened a share of the cells.
- R: [Subset and re-analyse in R](../../r_package/f_notebooks_tutorials/12_subset_and_reanalyse.ipynb)